The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there
(With "mv old_name new_name" you can rename the file) 

In [ ]:
# ALL THE CODE IN THIS EXERCISE WORKS FOR UBUNTU BASH.

nano script_ex3_1.sh

#!/bin/bash

mkdir students

# We download file and remane it.

wget "https://www.dropbox.com/scl/fi/bxv17nrbrl83vw6qrkiu9/LCP_22-23_students.csv?rlkey=47fakvatrtif3q3qw4q97p5b7&e=1" -O LCP_22-23_students.csv

rm wget-log

mv LCP_22-23_students.csv students/

# Control O for saving and control X for exit.

# We have to run the following to make the sript executable.

chmod +x script_ex3_1.sh

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

In [ ]:
cd students

grep "PoD" LCP_22-23_students.csv > PoD

grep "Physics" LCP_22-23_students.csv > Ph

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

In [ ]:
# Since the students names are in the second column, we have to use: 

for L in {A..Z}; do echo -n "$L: "; cut -d "," -f2 LCP_22-23_students.csv | grep -c "^$L"; done

# On where cut extracts columns (fields) from each line of a file.

# " -d "," " sets the delimiter to a comma because the file is a CSV (comma-separated values), this tells cut “columns are separated by commas”.

# "-f2" means "extract field (column) number 2".

1\.d Find out which is the letter with most counts.

In [ ]:
max=0;
letter="";

for L in {A..Z}; do c=$(cut -d "," -f2 LCP_22-23_students.csv | grep -ci "^$L"); 
if [ $c -gt $max ]; 
then max=$c; 
letter=$L; 
fi; 
done;

echo "Letter with most names: $letter ($max)"

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [ ]:
# "wc" gives number of lines, number of words, number of bytes and the file name. With -l we select only lines.

N=$(wc -l < LCP_22-23_students.csv) # "< LCP_22-23_students.csv" is used instead of just "LCP_22-23_students.csv" for taking just the number value.

# Code used:

for i in $(seq 2 $(wc -l < LCP_22-23_students.csv)); do g=$(( (i-2)%18+1 )); sed -n "${i}p" LCP_22-23_students.csv >> Group_$g; done

# "i" starts in 2 because 1st line is for titles.

# "g=$(( (i-2)%18+1 ))" uses 2 parenthesis becauses one is for the calculation and the second one to define it as a variable.

# "sed -n "${i}p" LCP_22-23_students.csv >> Group_$g" prints no line of LCP_22-23_students.csv unless "${i}p" line that is printed (included) in the file "Group_$g" that is automatically created if it does not exist.

# This overwrites the lists so if we want to make them again, we have to use " rm Group_* " to eliminate all the files named " Group_... ".

# We have to run the following to make the sript executable.

chmod +x script_ex3_1.sh

# Now for executing the script that will do the whole exercise we write:

./script_ex3_1.sh

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

In [ ]:
# ALL THE CODE IN THIS EXERCISE WORKS FOR UBUNTU BASH.

nano script_ex3_2.sh

# If the file data.csv isn't already created (we did it in the example inside the directory "test").

# echo "data.csv (content)" > data.csv

# If we want to copy the file already created, first we have to get out the specific directory "students" and then do the copy.

cp test/data.csv data.csv

# Now we make the copy we are asked for.

grep -v "^#" data.csv | sed -e "s/,//g" > data.txt

2\.b How many even numbers are there?

In [ ]:
# We will use " grep -Eo "[0-9]+" " to extract all numbers (of at least one digit) one per one in each line.

N=0; for x in $(grep -Eo "[0-9]+" data.txt); do if [ $x -ne 0 ] && [ $((x % 2)) -eq 0 ]; then N=$((N+1)); fi; done

echo "There are $N even numbers (excluding 0)."

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

In [ ]:
G=0; S=0; E=0; R=$(echo "scale=6; 100/2 * sqrt(3)" | bc); for i in $(seq 1 $(wc -l < data.txt)); do row=$(sed -n "${i}p" data.txt| tr -d '\r'); x=$(echo "$row" | cut -d ' ' -f1); y=$(echo "$row" | cut -d ' ' -f2); z=$(echo "$row" | cut -d ' ' -f3); D=$(echo "scale=6; sqrt($x*$x + $y*$y + $z*$z)" | bc); if (( $(echo "$D > $R" | bc) )); then G=$((G+1)); elif (( $(echo "$D < $R" | bc) )); then S=$((S+1)); else E=$((E+1)); fi; done

for i in $(seq 1 $(wc -l < data.txt)); do row=$(sed -n "${i}p" data.txt| tr -d '\r'); x=$(echo "$row" | cut -d ' ' -f4); y=$(echo "$row" | cut -d ' ' -f5); z=$(echo "$row" | cut -d ' ' -f6); D=$(echo "scale=6; sqrt($x*$x + $y*$y + $z*$z)" | bc); if (( $(echo "$D > $R" | bc) )); then G=$((G+1)); elif (( $(echo "$D < $R" | bc) )); then S=$((S+1)); else E=$((E+1)); fi; done

echo "There are $G greater, $S smaller and $E equal entries."
# It is important to notice that the code is repeteated for the first 3 numbers (x,y,z) and the last 3 (x',y',z').
# " row=$(sed -n "${i}p" data.txt| tr -d '\r') " has " | tr -d '\r' " that cleans Windows line endings (" z' ").
# Comparations are made with bc because those aren't integer numbers.

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [ ]:
for i in $(seq 2 $1); do > data_$i.txt; while read -r line; do new_line=""; for num in $line; do clean_num=$(echo "$num" | tr -d '\r'); new_num=$(echo "scale=6; $clean_num / $i" | bc); new_line="$new_line $new_num"; done; echo "$new_line" >> data_$i.txt; done < data.txt; done

# We have to run the following to make the sript executable.

chmod +x script_ex3_2.sh

# Now for executing the script that will do the whole exercise we write:

./script_ex3_2.sh NUMBER_OF_COPIES

# For removing the copies created:

rm data_*